# Step 7a · Backcasting 2026 → 2050: reach a 35 % bike share at least cost (Grenoble)

**Question:** how much must Grenoble invest, and when, to reach a **35 % predicted bike share in 2050**?

*Backcasting* starts from a target in the future and works backwards to the cheapest path that reaches it.
The network response to money comes from the **MLCC lookup table** of step 5 (budget → network metrics → share),
smoothed with PCHIP curves; no network computation is needed here. Two policies are compared:

* **terminal-only** — reach 35 % in 2050, with no intermediate milestones;
* **linear milestones** — progress linearly from today's predicted share to 35 % in 2050;

both with an illustrative maximum annual allocation (2 M€/year). For a CO₂ target instead of a share target,
see notebook `07b_backcasting_2050_co2_target.ipynb`.

| | |
|---|---|
| **Input** | `results/mlcc_lookup/grenoble_lookup_infrastructure_only.csv` (step 5) |
| **Output** | `results/backcasting/target_35pct/` |
| **Run time** | seconds |

In [1]:
# Setup: find the repository root and make the `cyclelts` package importable.
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "cyclelts").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy.interpolate import PchipInterpolator
from scipy.optimize import brentq
from IPython.display import display

from cyclelts import config

# Lookup table produced by notebooks/05_mlcc_lookup_table.ipynb
LOOKUP_DIR = config.LOOKUP_DIR
PREFERRED_CSV = LOOKUP_DIR / 'grenoble_lookup_infrastructure_only.csv'
FALLBACK_CSV = LOOKUP_DIR / 'grenoble_mlcc_lookup_1000.csv'
LOOKUP_CSV = PREFERRED_CSV if PREFERRED_CSV.exists() else FALLBACK_CSV
if not LOOKUP_CSV.exists():
    raise FileNotFoundError(
        f'Cannot find the lookup CSV: run notebooks/05_mlcc_lookup_table.ipynb first. '
        f'Tried: {PREFERRED_CSV} and {FALLBACK_CSV}'
    )

OUTPUT_DIR = config.BACKCASTING_DIR / 'target_35pct'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

START_YEAR, END_YEAR = 2026, 2050
TARGET_SHARE_PERCENT = 35.0
CAR_SHARE_2026 = 0.25
ANNUAL_ALLOCATION_CAP_EUR = 2_000_000.0  # Illustrative; adjust to your policy context

W_N, W_C = 0.40, 0.60
ALPHA, BETA, P_MAX = -3.40675, 5.01498, 0.50

raw = pd.read_csv(LOOKUP_CSV).sort_values('budget_allocated_eur')
needed = ['budget_allocated_eur','normalized_network_length','weighted_lcc_normalized']
assert all(c in raw for c in needed), f'Missing columns: {set(needed)-set(raw.columns)}'
raw = raw.drop_duplicates('budget_allocated_eur').reset_index(drop=True)
assert len(raw) >= 3
b = raw['budget_allocated_eur'].to_numpy(dtype=float)
length = raw['normalized_network_length'].to_numpy(dtype=float)
lcc = raw['weighted_lcc_normalized'].to_numpy(dtype=float)
assert np.isclose(b[0], 0) and np.all(np.diff(b)>0)
assert np.isfinite(length).all() and np.isfinite(lcc).all()
print(f'Loaded {len(raw)} lookup states from {LOOKUP_CSV.resolve()}')
print(f'Cumulative allocation range: €{b[0]:,.0f} – €{b[-1]:,.0f}')


Loaded 1000 lookup states from \results\mlcc_lookup\grenoble_lookup_infrastructure_only.csv
Cumulative allocation range: €0 – €38,595,734


## Continuous response from the two saved infrastructure metrics

We fit **PCHIP** separately to normalized network length and weighted LCC, then apply the same calibrated logistic equation. The original MLCC lookup values remain saved and can be compared to the surrogate. We optionally correct very small nonmonotonic numerical artifacts in the two metrics using isotonic regression; corrections are reported rather than hidden.

In [2]:
from sklearn.isotonic import IsotonicRegression

def to_percent(n_value, c_value):
    z = ALPHA + BETA * (W_N * np.asarray(n_value) + W_C * np.asarray(c_value))
    return 100 * P_MAX / (1 + np.exp(-z))

# For a cumulative-budget metamodel, require no deterioration of either infrastructure metric.
# If the raw data decrease, quantify the changes rather than silently assuming monotonicity.
length_iso = IsotonicRegression(increasing=True, out_of_bounds='clip').fit_transform(b, length)
lcc_iso = IsotonicRegression(increasing=True, out_of_bounds='clip').fit_transform(b, lcc)
print('Maximum isotonic correction, length:', float(np.max(np.abs(length_iso-length))))
print('Maximum isotonic correction, LCC:   ', float(np.max(np.abs(lcc_iso-lcc))))

length_fn = PchipInterpolator(b, length_iso, extrapolate=False)
lcc_fn = PchipInterpolator(b, lcc_iso, extrapolate=False)

def infrastructure_and_share(cumulative_budget_eur):
    q = np.asarray(cumulative_budget_eur, dtype=float)
    if np.any(q < b[0]-1e-7) or np.any(q > b[-1]+1e-7):
        raise ValueError('Budget outside the simulated MLCC lookup range')
    q = np.clip(q, b[0], b[-1])
    N = length_fn(q)
    C = lcc_fn(q)
    P = to_percent(N, C)
    return N, C, P

P0 = float(infrastructure_and_share(0.0)[2])
OTHER_FIXED = 1.0 - CAR_SHARE_2026 - P0 / 100.0
assert OTHER_FIXED >= 0
MAX_FEASIBLE_SHARE = min(100*P_MAX, 100*(1-OTHER_FIXED))
print(f'Baseline 2026 bike/two-wheeler share: {P0:.5f}%')
print(f'Baseline car share: {CAR_SHARE_2026*100:.2f}%')
print(f'Fixed walking + PT + other share: {OTHER_FIXED*100:.5f}%')
print(f'Max feasible bike share with nonnegative car share: {MAX_FEASIBLE_SHARE:.5f}%')
print(f'Max modeled bike share at lookup endpoint: {float(infrastructure_and_share(b[-1])[2]):.5f}%')
assert TARGET_SHARE_PERCENT <= MAX_FEASIBLE_SHARE + 1e-8, 'Target exceeds modal-share feasibility limit.'
assert TARGET_SHARE_PERCENT <= float(infrastructure_and_share(b[-1])[2]) + 1e-8, 'Target not reachable within 0–50% Bmax lookup.'

# Verify the metamodel against the actual MLCC points.
observed_p = to_percent(length, lcc)
fit_p = infrastructure_and_share(b)[2]
print('In-sample maximum share discrepancy (due to isotonic smoothing):',
      f'{np.max(np.abs(observed_p-fit_p)):.6f} percentage points')


Maximum isotonic correction, length: 4.440892098500626e-16
Maximum isotonic correction, LCC:    1.1102230246251565e-16
Baseline 2026 bike/two-wheeler share: 22.82022%
Baseline car share: 25.00%
Fixed walking + PT + other share: 52.17978%
Max feasible bike share with nonnegative car share: 47.82022%
Max modeled bike share at lookup endpoint: 39.79101%
In-sample maximum share discrepancy (due to isotonic smoothing): 0.000000 percentage points


## Invert the lookup: investment required for a target mode share

The mode-share response may contain plateaus. The inverse returns the **lowest** cumulative allocation that achieves a given target, within the continuous surrogate domain.

In [3]:
def minimum_budget_for_share(share_percent):
    share_percent = float(share_percent)
    if share_percent <= P0 + 1e-10:
        return 0.0
    if share_percent > float(infrastructure_and_share(b[-1])[2]) + 1e-9:
        raise ValueError(f'Unattainable target: {share_percent:.3f}%')
    # Identify the first lookup interval crossing the target; then solve PCHIP + logistic exactly.
    shares = np.asarray(infrastructure_and_share(b)[2])
    j = int(np.searchsorted(shares, share_percent, side='left'))
    j = min(max(j, 1), len(b)-1)
    if abs(shares[j] - share_percent) < 1e-9:
        return float(b[j])
    return float(brentq(lambda q: float(infrastructure_and_share(q)[2])-share_percent,
                        b[j-1], b[j], xtol=1e-4))

B_REQUIRED = minimum_budget_for_share(TARGET_SHARE_PERCENT)
print(f'Minimum cumulative allocated budget for {TARGET_SHARE_PERCENT:.1f}%: €{B_REQUIRED:,.0f}')
print(f'As a fraction of simulated 50%-Bmax endpoint: {100*B_REQUIRED/b[-1]:.2f}%')


Minimum cumulative allocated budget for 35.0%: €16,446,797
As a fraction of simulated 50%-Bmax endpoint: 42.61%


## Optimize annual allocations with carryover

For a specified sequence of annual share requirements, we first compute the **minimum funding that must have been allocated by each year**. We then work backward from 2050 to ensure annual allocation limits can be met. This is a minimum-cost, latest-feasible schedule for nondecreasing targets and a nondecreasing budget response.

Unlike the expensive MLCC calculation, the annual model is fast. The prior MLCC lookup was already constructed with unspent funds carried forward. We estimate *actual spending* from the lookup table and show the unspent balance separately. No money disappears.

In [4]:
years = np.arange(START_YEAR, END_YEAR+1)
T = len(years)

def plan_scenario(name, target_path):
    target_path = np.asarray(target_path, dtype=float)
    if len(target_path) != T or np.any(np.diff(target_path)<-1e-9):
        raise ValueError('Target trajectory must be nondecreasing and match the planning horizon')
    minimum_needed = np.array([minimum_budget_for_share(p) for p in target_path])
    budget = minimum_needed.copy()
    # Work backward: funds required in year t so next year's allocation cap can be met.
    for t in range(T-2,-1,-1):
        budget[t] = max(budget[t], budget[t+1]-ANNUAL_ALLOCATION_CAP_EUR)
    if budget[0] > 1e-5:
        raise ValueError(f'{name}: annual budget cap too small; extra allocation before 2027 needed.')
    budget[0] = 0.0
    increments = np.diff(budget, prepend=0)
    if np.any(increments < -1e-5) or np.any(increments > ANNUAL_ALLOCATION_CAP_EUR+1e-5):
        raise AssertionError('Invalid annual allocation trajectory')
    N,C,P = infrastructure_and_share(budget)
    assert np.all(P+1e-7 >= target_path)
    # Actual spending is recorded by the original MLCC lookup; use piecewise linear
    # interpolation because 'actual_spent_eur' can be stepwise at budget levels.
    if 'actual_spent_eur' in raw:
        cumulative_spent = np.interp(budget, b, raw['actual_spent_eur'].to_numpy(dtype=float))
        cumulative_spent = np.minimum(np.maximum.accumulate(cumulative_spent), budget)
    else:
        cumulative_spent = np.full_like(budget, np.nan)
    car = 100*(1-OTHER_FIXED)-P
    return pd.DataFrame({
        'scenario': name, 'year': years,
        'target_share_percent': target_path,
        'annual_allocated_eur': increments,
        'cumulative_allocated_eur': budget,
        'cumulative_spent_eur_approx': cumulative_spent,
        'unspent_balance_eur_approx': budget-cumulative_spent,
        'normalized_network_length': N,
        'weighted_lcc_normalized': C,
        'predicted_bike_share_percent': P,
        'predicted_car_share_percent': car,
    })

terminal_targets = np.r_[np.repeat(P0,T-1), TARGET_SHARE_PERCENT]
linear_targets = np.linspace(P0,TARGET_SHARE_PERCENT,T)
terminal_df = plan_scenario('Terminal target only', terminal_targets)
linear_df = plan_scenario('Linear annual milestones', linear_targets)
results = pd.concat([terminal_df, linear_df], ignore_index=True)
results.to_csv(OUTPUT_DIR/'grenoble_annual_backcasting_35pct.csv',index=False)

summary = results.groupby('scenario',sort=False).agg(
    total_allocated_eur=('annual_allocated_eur','sum'),
    share_2050_percent=('predicted_bike_share_percent','last'),
    car_2050_percent=('predicted_car_share_percent','last'),
    max_annual_allocation_eur=('annual_allocated_eur','max'),
).reset_index()
display(summary.round(4))
display(results[results.year.isin([2026,2030,2035,2040,2045,2050])].round(4))
print('Saved:', (OUTPUT_DIR/'grenoble_annual_backcasting_35pct.csv').resolve())


,scenario,total_allocated_eur,share_2050_percent,car_2050_percent,max_annual_allocation_eur
0,Terminal target only,1.644680e+07,35.0,12.8202,2000000.0
1,Linear annual milestones,1.644680e+07,35.0,12.8202,2000000.0


,scenario,year,target_share_percent,annual_allocated_eur,cumulative_allocated_eur,cumulative_spent_eur_approx,unspent_balance_eur_approx,normalized_network_length,weighted_lcc_normalized,predicted_bike_share_percent,predicted_car_share_percent
0,Terminal target only,2026,22.8202,0.000000e+00,0.000000e+00,0.000000e+00,0.0000,1.0,0.4074,22.8202,25.0000
4,Terminal target only,2030,22.8202,0.000000e+00,0.000000e+00,0.000000e+00,0.0000,1.0,0.4074,22.8202,25.0000
9,Terminal target only,2035,22.8202,0.000000e+00,0.000000e+00,0.000000e+00,0.0000,1.0,0.4074,22.8202,25.0000
14,Terminal target only,2040,22.8202,0.000000e+00,0.000000e+00,0.000000e+00,0.0000,1.0,0.4074,22.8202,25.0000
19,Terminal target only,2045,22.8202,2.000000e+06,6.446797e+06,6.420273e+06,26524.0544,1.0,0.6363,31.2872,16.5330
24,Terminal target only,2050,35.0000,2.000000e+06,1.644680e+07,1.644204e+07,4754.1533,1.0,0.7471,35.0000,12.8202
25,Linear annual milestones,2026,22.8202,0.000000e+00,0.000000e+00,0.000000e+00,0.0000,1.0,0.4074,22.8202,25.0000
29,Linear annual milestones,2030,24.8502,5.167028e+03,1.805775e+05,1.543231e+05,26254.3458,1.0,0.4615,24.8502,22.9700
34,Linear annual milestones,2035,27.3876,1.445066e+06,2.616655e+06,2.608324e+06,8330.4221,1.0,0.5292,27.3876,20.4326
39,Linear annual milestones,2040,29.9251,4.567973e+05,4.670695e+06,4.667788e+06,2907.6931,1.0,0.5982,29.9251,17.8951


Saved: \results\backcasting\target_35pct\grenoble_annual_backcasting_35pct.csv


## Plot the dynamic pathways

Blue: terminal-only target (may postpone investments). Orange: linear annual targets (requires earlier improvements). Both reach the same 2050 modal share and require the same minimum **cumulative allocated** budget under this model. Their annual schedules differ.

In [5]:
for ycol, label, scale in [
    ('annual_allocated_eur','Annual allocated budget (€ million)',1e6),
    ('cumulative_allocated_eur','Cumulative allocated budget (€ million)',1e6),
    ('predicted_bike_share_percent','Predicted bicycle mode share (%)',1),
    ('predicted_car_share_percent','Residual car mode share (%)',1),
    ('unspent_balance_eur_approx','Unspent funding balance (€ million, approximate)',1e6),
]:
    fig=go.Figure()
    for name, color in [('Terminal target only','#2563eb'),('Linear annual milestones','#f97316')]:
        d=results[results.scenario==name]
        fig.add_trace(go.Scatter(x=d.year,y=d[ycol]/scale,mode='lines+markers',
                                 name=name,line=dict(color=color,width=2.5)))
    if ycol=='predicted_bike_share_percent':
        fig.add_hline(y=TARGET_SHARE_PERCENT,line_dash='dot',line_color='black',
                      annotation_text='2050 target: 35%')
    fig.update_layout(title=f'Grenoble 2026–2050 — {label}',xaxis_title='Year',
                      yaxis_title=label,template='plotly_white',height=450)
    fig.show()


## Plot the underlying lookup and 2050 threshold

In [6]:
dense=np.linspace(0,b[-1],2000)
N,C,P=infrastructure_and_share(dense)
fig=go.Figure()
fig.add_trace(go.Scatter(x=b/1e6,y=observed_p,mode='lines',
                         name='MLCC lookup (1,000 states)',line=dict(color='#475569',width=1.5)))
fig.add_trace(go.Scatter(x=dense/1e6,y=P,mode='lines',name='PCHIP of N and LCC + logistic share',
                         line=dict(color='#0891b2',width=2)))
fig.add_hline(y=TARGET_SHARE_PERCENT,line_dash='dash',annotation_text='35% target')
fig.add_vline(x=B_REQUIRED/1e6,line_dash='dash',line_color='#f97316',
              annotation_text=f'€{B_REQUIRED/1e6:.2f}M required')
fig.update_layout(title='Grenoble — Infrastructure lookup and 35% target',
                  xaxis_title='Cumulative allocated investment (€ million)',
                  yaxis_title='Predicted bicycle mode share (%)',template='plotly_white',height=500)
fig.show()


## Important interpretation

- **2026**: predicted bike/two-wheeler share ≈22.82%, car share assumed to be 25%. Fixed other modes sum to the residual ≈52.18%.
- **2050 target**: bicycle/two-wheeler share ≥35%, giving a modeled car share of approximately **12.82%**. This is a percentage-point shift in the *mode-share accounting*, not yet a measured change in vehicle-kilometres or CO₂.
- **Carryover**: MLCC's lookup is indexed by *cumulative allocated funds*, includes its original unspent balance, and does not discard money. The planning model additionally displays the approximate unspent balance estimated from the saved lookup.
- **Approximation**: PCHIP interpolates normalized length and weighted LCC separately, and their outputs determine bike share. Surrogate predictions need not correspond to a physically buildable partial road upgrade for arbitrary intermediate budgets.
- **Temporal interpretation**: annual share milestones are a policy assumption, not an emissions calculation; the terminal-only problem alone does not favor early infrastructure spending.
- **Annual spending cap**: the code constrains *new allocated money* each year; if a real policy imposes a limit on *actual construction spending*, or annual work capacity, an additional project-delivery model would be needed. `cumulative_spent_eur_approx` is interpolated from the saved MLCC lookup, not a new MLCC execution.
- **Target validity**: the 50% logistic maximum and nonnegative residual car share are both checked. The underlying mode-share calibration used an INSEE two-wheeler measure, so label it carefully when discussing bicycle-only shifts.

## Next: a CO₂ target

An emissions-based version of this backcasting (car share, car distance, CO₂ per vehicle-km, discounting, exact
optimisation on the 1000 MLCC states) is in **`07b_backcasting_2050_co2_target.ipynb`**. Its scenario
*"Original notebook"* (ξ = 1, r_BC = 1) keeps the simple assumption that every bike trip replaces an average car trip.